依據 Profiling 結果 → 套用 Business Rules → 建立 Analysis Dataset → 供後續 SQL 與 Dashboard 使用

In [56]:
# 第一步:讀取原始資料
import pandas as pd

df = pd.read_excel(
    "../data/raw/Online Retail.xlsx",
    engine="openpyxl"
)

# df.head()
print(f"Raw Data Rows: {len(df):,}")

Raw Data Rows: 541,909


In [57]:
# 第二步: 套用 Rule 1 - 取消訂單
df["is_cancellation"] = df["InvoiceNo"].astype(str).str.startswith("C")
df["is_cancellation"].value_counts()

is_cancellation
False    532621
True       9288
Name: count, dtype: int64

In [58]:
sales_df = df[~df["is_cancellation"]].copy()
print(f"After Rule 1: {len(sales_df):,}")

After Rule 1: 532,621


In [59]:
# 第三步: 套用 Rule 2 - Non-Sales Transactions
sales_df["is_non_sales"] = sales_df["UnitPrice"] <= 0
sales_df["is_non_sales"].value_counts()

is_non_sales
False    530104
True       2517
Name: count, dtype: int64

In [60]:
sales_df_non_price = sales_df[~sales_df["is_non_sales"]].copy()
print(f"After Rule 2: {len(sales_df_non_price):,}")

After Rule 2: 530,104


In [61]:
# 第四步: 套用 Rule 3 - Duplicate Records
sales_df_non_price.duplicated().sum()

np.int64(5226)

In [62]:
sales_df_no_duplicates = sales_df_non_price.drop_duplicates()
print(f"Before Rule 3: {len(sales_df_non_price):,}")
print(f"After Rule 3: {len(sales_df_no_duplicates):,}")

Before Rule 3: 530,104
After Rule 3: 524,878


In [63]:
remove_Rows = len(sales_df_non_price) - len(sales_df_no_duplicates)
print(f"Removed Duplicates Rows: {remove_Rows:,}")

Removed Duplicates Rows: 5,226


In [64]:
# # 第五步: 統計清理後的資料筆數, 建立 Data Cleaning  Summary
summary = pd.DataFrame({
    "Step" : [
        "Raw Data",
        "After Rule 1 - (Remove Cancellations)",
        "After Rule 2 - (Remove Non-Sales)",
        "After Rule 3 - (Remove Duplicates)"],
    "Rows" : [
        len(df),
        len(sales_df),
        len(sales_df_non_price),
        len(sales_df_no_duplicates)]
})

summary

,Step,Rows
0,Raw Data,541909
1,After Rule 1 - (Remove Cancellations),532621
2,After Rule 2 - (Remove Non-Sales),530104
3,After Rule 3 - (Remove Duplicates),524878


In [77]:
# 第六步: 輸出 Clean Dataset
clean_sales = sales_df_no_duplicates.drop(columns=["is_cancellation", "is_non_sales"])
clean_sales["CustomerID"] = clean_sales["CustomerID"].astype("Int64")
clean_sales.to_csv("../data/processed/clean_sales.csv", index=False)


In [66]:
import os
os.path.exists("../data/processed/clean_sales.csv")

True

In [67]:
clean_sales.shape

(524878, 8)

In [68]:
clean_sales.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [78]:
clean_sales.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
